In [1]:
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.utils import resample

print("Libraries loaded!")

Libraries loaded!


In [2]:
train_df = pd.read_csv("../dataset/KDDTrain+.csv")

print("Original Dataset Shape:", train_df.shape)
train_df.head()

Original Dataset Shape: (50000, 25)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,num_failed_logins,num_compromised,is_host_login,is_guest_login,...,source_port,destination_port,session_time,timestamp,previous_attack_count,risk_score,trust_score,attack_category,anomaly_score,attack_success
0,241,tcp,dns,REJ,3738,1004,0,0,0,1,...,61049,80,241,2026-03-6 18:7:4,19,0.24,0.76,normal,0.30,0
1,292,tcp,ntp,REJ,3484,7859,5,1,1,0,...,29523,80,292,2026-03-10 23:44:51,5,0.01,0.99,normal,0.07,0
2,289,udp,dns,S0,3012,1879,2,3,0,1,...,59724,53,289,2026-03-26 11:33:28,5,0.17,0.83,normal,0.24,0
3,286,udp,http,REJ,4329,4676,0,2,0,0,...,39272,22,286,2026-03-28 12:59:24,14,0.27,0.73,normal,0.39,0
4,265,icmp,dns,S0,2195,6812,2,2,1,1,...,37787,53,265,2026-03-5 0:47:32,20,0.04,0.96,normal,0.17,0


In [3]:
print("Class Distribution BEFORE balancing:\n")
print(train_df["class"].value_counts())

Class Distribution BEFORE balancing:

class
normal          20000
neptune         15000
smurf           10000
guess_passwd     5000
Name: count, dtype: int64


In [4]:
from sklearn.utils import resample

# Find majority class
class_counts = train_df["class"].value_counts()

majority_class = class_counts.idxmax()
minority_size = class_counts.min()

df_balanced = []

for cls in class_counts.index:
    df_class = train_df[train_df["class"] == cls]

    df_resampled = resample(
        df_class,
        replace=True,                 # 🔥 allows up/down sampling
        n_samples=minority_size,      # make all equal
        random_state=42
    )

    df_balanced.append(df_resampled)

# Combine all
train_df = pd.concat(df_balanced)

print("Class Distribution AFTER balancing:\n")
print(train_df["class"].value_counts())

Class Distribution AFTER balancing:

class
normal          5000
neptune         5000
smurf           5000
guess_passwd    5000
Name: count, dtype: int64


In [5]:
y = train_df["class"]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

with open("../model/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("Label encoding done!")

Label encoding done!


In [6]:
print("\nTraining distribution (encoded):")
print(pd.Series(y).value_counts())


Training distribution (encoded):
2    5000
1    5000
3    5000
0    5000
Name: count, dtype: int64


In [7]:
X = train_df.drop(["class", "classnum"], axis=1, errors="ignore")

drop_cols = ["source_ip", "destination_ip", "timestamp"]
X = X.drop(columns=[col for col in drop_cols if col in X.columns])

categorical_cols = ["protocol_type", "service", "flag", "attack_category"]
existing_cats = [col for col in categorical_cols if col in X.columns]

X = pd.get_dummies(X, columns=existing_cats)

X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

feature_columns = X.columns.tolist()

with open("../model/feature_columns.pkl", "wb") as f:
    pickle.dump(feature_columns, f)

print("Feature processing done!")
print("Feature shape:", X.shape)

Feature processing done!
Feature shape: (20000, 30)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (16000, 30)
Test shape: (4000, 30)


In [9]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

with open("../model/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Scaling completed!")

Scaling completed!


In [10]:
# ================================
# 🔥 IMPROVED MODELS (IMPORTANT)
# ================================

rf = RandomForestClassifier(
    n_estimators=300,       # 🔥 increased
    max_depth=15,           # 🔥 deeper
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42
)

svm = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",
    max_iter=2000,          # 🔥 more training
    random_state=42
)

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),   # 🔥 deeper network
    max_iter=300,
    random_state=42
)

ada = AdaBoostClassifier(
    n_estimators=120,       # 🔥 more estimators
    random_state=42
)

print("Improved models initialized!")

Improved models initialized!


In [11]:
rf.fit(X_train, y_train)
svm.fit(X_train, y_train)
mlp.fit(X_train, y_train)
ada.fit(X_train, y_train)

print("Models trained!")

d:\anaconda\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Models trained!


In [12]:
rf_acc = accuracy_score(y_test, rf.predict(X_test))
svm_acc = accuracy_score(y_test, svm.predict(X_test))
mlp_acc = accuracy_score(y_test, mlp.predict(X_test))
ada_acc = accuracy_score(y_test, ada.predict(X_test))

print("RF Accuracy :", rf_acc)
print("SVM Accuracy:", svm_acc)
print("MLP Accuracy:", mlp_acc)
print("ADA Accuracy:", ada_acc)

RF Accuracy : 0.999
SVM Accuracy: 0.99325
MLP Accuracy: 0.99525
ADA Accuracy: 0.75


In [13]:
accuracies = {
    "rf": rf_acc,
    "svm": svm_acc,
    "mlp": mlp_acc,
    "ada": ada_acc
}

total = sum(accuracies.values())
weights = {model: acc / total for model, acc in accuracies.items()}

print("Model Weights:", weights)

with open("../model/model_weights.pkl", "wb") as f:
    pickle.dump(weights, f)

Model Weights: {'rf': 0.26729096989966555, 'svm': 0.265752508361204, 'mlp': 0.2662876254180602, 'ada': 0.20066889632107024}


In [14]:
pickle.dump(rf, open("../model/rf_model.pkl", "wb"))
pickle.dump(svm, open("../model/svm_model.pkl", "wb"))
pickle.dump(mlp, open("../model/mlp_model.pkl", "wb"))
pickle.dump(ada, open("../model/ada_model.pkl", "wb"))

print("All models saved successfully!")

All models saved successfully!
